In [1]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd
pd.set_option('display.max_rows', 8)
!date

Wed 07 Aug 2024 05:16:52 PM PDT


# Mean deaths and stillbirths averted by adding folate by wealth quintile


In [2]:
import vivarium_inputs
import db_queries
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [3]:
location = "india"
vehicle = "rice"
intervention_scenario = "intervention"

In [4]:
# Parameters
location = "nigeria"
vehicle = "bouillon"
intervention_scenario = "intervention"


In [5]:
asfr = vivarium_inputs.get_measure(gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location.title(), years=2022).value

In [6]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel("parameter")

In [7]:
# Scale ASFR in each category down proportionally to the scale-down in TFR forecasted from GBD 2017
if location == "india":
    asfr_2030_to_2022_ratio = (
        1.61 / 1.91 # http://ihmeuw.org/6j8s
    )
elif location == "nigeria":
    asfr_2030_to_2022_ratio = (
        4.43 / 4.96 # http://ihmeuw.org/6jqx
    )
elif location == "ethiopia":
    asfr_2030_to_2022_ratio = (
        3.27 / 4.10 # http://ihmeuw.org/6j7d
    )

asfr = asfr * asfr_2030_to_2022_ratio
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end
Nigeria   Female  10.0       15.0     2022        2023        0.002739
                  15.0       20.0     2022        2023        0.067988
                  20.0       25.0     2022        2023        0.178627
                  25.0       30.0     2022        2023        0.195842
                                                                ...   
                  35.0       40.0     2022        2023        0.119953
                  40.0       45.0     2022        2023        0.062535
                  45.0       50.0     2022        2023        0.030010
                  50.0       55.0     2022        2023        0.002782
Name: value, Length: 9, dtype: float64

In [8]:
asfr = asfr.reset_index().assign(year_start=2030, year_end=2031).set_index(asfr.index.names).value
asfr.sort_values()

location  sex     age_start  age_end    year_start  year_end
Nigeria   Female  0.000000   0.019178   2030        2031        0.000000
          Male    0.076712   0.500000   2030        2031        0.000000
                  0.500000   1.000000   2030        2031        0.000000
                  1.000000   2.000000   2030        2031        0.000000
                                                                  ...   
          Female  35.000000  40.000000  2030        2031        0.119953
                  20.000000  25.000000  2030        2031        0.178627
                  30.000000  35.000000  2030        2031        0.179030
                  25.000000  30.000000  2030        2031        0.195842
Name: value, Length: 50, dtype: float64

In [9]:
from vivarium_inputs import utilities
from vivarium_inputs.utility_data import get_location_id
from vivarium_gbd_access.gbd import get_age_group_id, SEX, RELEASE_IDS

In [10]:
def get_population_future(location, year):
    # Cobbled together from pieces of vivarium_inputs and vivarium_gbd_access
    # TODO: vivarium_inputs should be able to get forecasted pop!
    location_id = get_location_id(location)
    year_id = year
    data = db_queries.get_population(
        age_group_id=get_age_group_id(),
        location_id=location_id,
        year_id=year_id,
        sex_id=SEX.MALE + SEX.FEMALE + SEX.COMBINED,
        release_id=RELEASE_IDS.GBD_2021,
        forecasted_pop=True,
    )
    data = utilities.normalize_sex(data.drop("run_id", axis="columns").rename(columns={"population": "value"}), fill_value=None, cols_to_fill=utilities.DRAW_COLUMNS)
    data = utilities.reshape(data, ["value"])
    data = utilities.scrub_gbd_conventions(data, location)
    data = utilities.split_interval(data, interval_column="age", split_column_prefix="age")
    data = utilities.split_interval(data, interval_column="year", split_column_prefix="year")
    return utilities.sort_hierarchical_data(data)


In [11]:
pop = get_population_future(location.title(), 2030).value.reindex(asfr.index)
pop

location  sex     age_start  age_end     year_start  year_end
Nigeria   Female  0.000000   0.019178    2030        2031         76752.699967
                  0.019178   0.076712    2030        2031        227071.059445
                  0.076712   0.500000    2030        2031                  NaN
                  0.500000   1.000000    2030        2031                  NaN
                                                                     ...      
          Male    80.000000  85.000000   2030        2031        354428.130049
                  85.000000  90.000000   2030        2031        177910.014136
                  90.000000  95.000000   2030        2031         62189.478603
                  95.000000  125.000000  2030        2031         16933.452258
Name: value, Length: 50, dtype: float64

In [12]:
# Forecasted population does not have younger ages, but luckily none of these are WRA
assert (pop.index.get_level_values("age_end")[pop.isna()] < 10).all()
pop[pop.isna()]

location  sex     age_start  age_end  year_start  year_end
Nigeria   Female  0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
          Male    0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
Name: value, dtype: float64

In [13]:
pop = pop.fillna(0)

In [14]:
n_births = (pop * asfr).sum()
n_births

9859503.033205396

In [15]:
sbr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.stillbirth_to_live_birth_ratio,
    "estimate",
    location.title(),
    years=2022,
).value
sbr

location  year_start  year_end  parameter  
Nigeria   2022        2023      lower_value    0.037194
                                mean_value     0.037194
                                upper_value    0.037194
Name: value, dtype: float64

In [16]:
sbr = sbr[sbr.index.get_level_values("parameter") == "mean_value"].droplevel("parameter")
sbr

location  year_start  year_end
Nigeria   2022        2023        0.037194
Name: value, dtype: float64

In [17]:
sbr = sbr.values[0]

In [18]:
births_and_stillbirths = n_births + n_births * sbr
births_and_stillbirths / 1e6

10.22621267373506

In [19]:
if location == "india":
    dist_births_and_stillbirths_by_wealth = pd.Series( # from file:///J:/DATA/DHS_PROG_DHS/IND/2019_2021/IND_DHS7_2019_2021_REP_FINAL_Y2022M05D11.PDF
        dict(                          # Table 8.4 Perinatal mortality -- using "Number of pregnancies of 7 or more months' duration" as a proxy
            lowest=56_979,
            second=50_335,
            middle=45_189,
            fourth=42_611,
            highest=36_290,
        )
    )
elif location == "nigeria":
    dist_births_and_stillbirths_by_wealth = pd.Series( # from file:///J:/DATA/DHS_PROG_DHS/NGA/2018/NGA_DHS7_2018_REP_QUEST_Y2019M11D05.PDF
        dict(                          # Table 8.4 Perinatal mortality
            lowest=7_712,
            second=7_886,
            middle=7_139,
            fourth=6_328,
            highest=5_558,
        )
    )
elif location == "ethiopia":
    dist_births_and_stillbirths_by_wealth = pd.Series( # from file:///J:/DATA/DHS_PROG_DHS/ETH/2016/ETH_DHS7_2016_REP_QUEST_Y2017M08D15.PDF
        dict(                          # Table 8.4 Perinatal mortality
            lowest=2_645,
            second=2_516,
            middle=2_290,
            fourth=2_018,
            highest=1_592,
        )
    )

s_births = n_births * dist_births_and_stillbirths_by_wealth / dist_births_and_stillbirths_by_wealth.sum()
s_births

lowest     2.196126e+06
second     2.245676e+06
middle     2.032955e+06
fourth     1.802008e+06
highest    1.582737e+06
dtype: float64

In [20]:
s_births_and_stillbirths_by_wealth = births_and_stillbirths * dist_births_and_stillbirths_by_wealth / dist_births_and_stillbirths_by_wealth.sum()
s_births_and_stillbirths_by_wealth

lowest     2.277808e+06
second     2.329201e+06
middle     2.108567e+06
fourth     1.869031e+06
highest    1.641605e+06
dtype: float64

In [21]:
# http://ihmeuw.org/6jr2 -- extracted from GBD Foresight, count of NTD deaths for under-1 year olds
if location == "india":
    ntd_deaths = 4_273.37
elif location == "nigeria":
    ntd_deaths = 5_373.52
elif location == "ethiopia":
    ntd_deaths = 1_883.76


ntd_death_rate = ntd_deaths / n_births
10_000 * ntd_death_rate

5.450092141462662

In [22]:
# Assumed does not vary by wealth
champs_ntd_stillbirth_per_livebirth = 51 / (69-51)
ntd_stillbirths = ntd_deaths * champs_ntd_stillbirth_per_livebirth
10_000 * (ntd_deaths + ntd_stillbirths) / n_births # ntd rate, compare with 41 per 10,000 from Bhide et al https://pubmed.ncbi.nlm.nih.gov/23873811/

20.89201987560687

We could not find a good source for folate intake by wealth in India, or even a representative source for overall folate intake.  [This paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10755415/pdf/S1368980023002112a.pdf) has an overall number of 220 mcg/day for women, and since it is not too different from the values we found in Ethiopia and Nigeria, we are going to use it for now. (It also has standard deviation of 50, which we can use when we introduce heterogeneity)

In [23]:
if location == "india":
    s_baseline_folate = pd.Series(
        dict(
            lowest=220,
            second=220,
            middle=220,
            fourth=220,
            highest=220, # NRV is 400 mcg/day
        )
    )
elif location == "nigeria":
    # Table 95 of NFCMS 2021 Report
    s_baseline_folate = pd.Series(
        dict(
            lowest=189,
            second=198,
            middle=197,
            fourth=203,
            highest=208, # NRV is 400 mcg/day
        )
    )  # how can we use this to estimate disparities in NTD?
elif location == "ethiopia":
    # Table 6 of https://cdn.nutrition.org/article/S2475-2991%2824%2901728-1/fulltext
    s_baseline_folate = pd.Series(
        dict(
            lowest=166,
            second=152,
            middle=137,
            fourth=350,
            highest=469, # NRV is 400 mcg/day
        )
    )  # how can we use this to estimate disparities in NTD?

In [24]:
if location == "india":
    s_dist_deaths_by_wealth = pd.Series( # Table 7.9 on page 201 of the CNNS report has RBC folate deficiency rates;
        dict(                            # it includes wealth stratification, but has a very low threshold for insufficiency 
            lowest=1,                        # so I am assuming that most everyone is in the danger zone for low folate
            second=1,
            middle=1,
            fourth=1,
            highest=1,
        )       
    )
elif location == "nigeria":
    s_dist_deaths_by_wealth = pd.Series( # assume same rate for all, for now;
        dict(                          # can CHAMPS offer more detail?  Need to infer wealth somehow
            lowest=1,
            second=1,
            middle=1,
            fourth=1,
            highest=1,
        )       
    )
elif location == "ethiopia":
    s_dist_deaths_by_wealth = pd.Series(
        dict(                          # supplementation studies don't make this easy, but here is a guess
            lowest=5,
            second=5,
            middle=5,
            fourth=2,
            highest=1,
        )
    )
    s_dist_deaths_by_wealth /= s_dist_deaths_by_wealth.mean()

s_dist_deaths_by_wealth

lowest     1
second     1
middle     1
fourth     1
highest    1
dtype: int64

In [25]:
s_ntd_death_rate = ntd_deaths/n_births * s_dist_deaths_by_wealth
10_000 * s_ntd_death_rate

lowest     5.450092
second     5.450092
middle     5.450092
fourth     5.450092
highest    5.450092
dtype: float64

In [26]:
s_ntd_death_count = s_ntd_death_rate * s_births_and_stillbirths_by_wealth
s_ntd_death_count

lowest     1241.426439
second     1269.435801
middle     1149.188712
fourth     1018.639329
highest     894.689853
dtype: float64

In [27]:
s_ntd_death_count.sum(), ntd_deaths # should be similar

(5573.380133004933, 5373.52)

In [28]:
s_ntd_stillbirth_count = s_ntd_death_count * champs_ntd_stillbirth_per_livebirth
s_ntd_stillbirth_count

lowest     3517.374910
second     3596.734769
middle     3256.034684
fourth     2886.144765
highest    2534.954583
dtype: float64

In [29]:
s_ntd_death_or_stillbirth_count = s_ntd_death_count + s_ntd_stillbirth_count
s_ntd_death_or_stillbirth_count

lowest     4758.801348
second     4866.170569
middle     4405.223395
fourth     3904.784094
highest    3429.644436
dtype: float64

In [30]:
def backcalc_rbc(ntd_risk, method):
    """
    ln (odds of NTD risk) = 1.6563 − 1.2193 × ln (RBC) (Daly et al, 1995)
    ln (odds of NTD risk) = 4.57 − 1.70 × ln (RBC) (Crider et al, 2014)
    """
    
    odds = ntd_risk / (1 - ntd_risk)
    ln_odds = np.log(odds)
    if method == 'daly':
        neg_ln_rbc = (ln_odds - 1.6563) / 1.2193
    elif method == 'crider':
        neg_ln_rbc = (ln_odds - 4.57) / 1.70
    rbc = np.exp(-neg_ln_rbc)
    return rbc
backcalc_rbc(s_ntd_death_or_stillbirth_count / s_births, 'daly')

lowest     594.518262
second     594.518262
middle     594.518262
fourth     594.518262
highest    594.518262
dtype: float64

In [31]:
backcalc_rbc(s_ntd_death_or_stillbirth_count / s_births, 'crider')  # compare with CNNS, https://www.unicef.org/india/media/2646/file/CNNS-report.pdf in Table 7.9

lowest     542.106217
second     542.106217
middle     542.106217
fourth     542.106217
highest    542.106217
dtype: float64

In [32]:
if location == "india":
    assert vehicle == "rice"
    s_daily_vehicle = pd.Series( # Zeb and Alix analysis of HCES
        dict(
            lowest=213.570675, # grams
            second=175.027768,
            middle=163.699804,
            fourth=163.363078,
            highest=127.874174,
        )
    )
elif location == "nigeria":
    assert vehicle == "bouillon"
    s_daily_vehicle = pd.Series( # NFCMS 2021, Table 170. Usual intake of Bouillon (raw weight, grams) of women
        dict(                
            lowest=8.4, # grams
            second=8.0,
            middle=5.9,
            fourth=4.9,
            highest=4.6,
        )
    )
elif location == "ethiopia":
    assert vehicle == "salt"
    s_daily_vehicle = pd.Series( # Dememoz Woldegebreal, personal communication of analysis
        dict(                 # of 2013 Ethiopian National Food Consumption Survey (ENFCS)
            lowest=7.5467, # grams
            second=6.3605,
            middle=6.5508,
            fourth=6.5491,
            highest=6.5406,
        )
    ) * 0.90 # Saje et al 2024 assume 90% of total salt consumption comes from discretionary salt and manufactured food items (cites James et al 1987 )

In [33]:
baseline_concentration_mcg_per_gram = pd.read_csv(f'../0100_data_prep/results/folate/{vehicle}/baseline_fortification/concentration/{location}.csv')
assert (baseline_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert baseline_concentration_mcg_per_gram.value.nunique() == 1
baseline_concentration_mcg_per_gram = baseline_concentration_mcg_per_gram.value.iloc[0]
baseline_concentration_mcg_per_gram

0.0

In [34]:
intervention_concentration_mcg_per_gram = pd.read_csv(f'../0100_data_prep/results/folate/{intervention_scenario}/intervention_fortification/concentration/{location}.csv')
assert (intervention_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert intervention_concentration_mcg_per_gram.value.nunique() == 1
intervention_concentration_mcg_per_gram = intervention_concentration_mcg_per_gram.value.iloc[0]
intervention_concentration_mcg_per_gram

24.0

In [35]:
eff_fort_baseline_path = f'../0100_data_prep/results/folate/{vehicle}/baseline_fortification/effective_coverage/{location}.csv'
eff_fort_intervention_path = f'../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv'

df_eff_fort_baseline = pd.read_csv(eff_fort_baseline_path)
assert (df_eff_fort_baseline.vehicle_name == vehicle).all()
df_eff_fort_intervention = pd.read_csv(eff_fort_intervention_path)
assert (df_eff_fort_intervention.vehicle_name == vehicle).all()

In [36]:
# NOTE: Using DHS definition of WRA
population = pd.read_csv(f'../0100_data_prep/results/population/stratified/{location}.csv').groupby(["sex", "age_start", "age_end", "wealth_quintile"]).value.sum().reset_index()
population = population[(population.sex == "Female") & (population.age_start >= 15) & (population.age_end <= 50)]
population

,sex,age_start,age_end,wealth_quintile,value
40,Female,15.0,20.0,fourth,3.035520e+06
41,Female,15.0,20.0,highest,2.878370e+06
42,Female,15.0,20.0,lowest,2.337059e+06
43,Female,15.0,20.0,middle,2.786835e+06
...,...,...,...,...,...
71,Female,45.0,50.0,highest,1.081196e+06
72,Female,45.0,50.0,lowest,7.310809e+05
73,Female,45.0,50.0,middle,8.097988e+05
74,Female,45.0,50.0,second,7.610023e+05


In [37]:
if "sex" in df_eff_fort_baseline.columns:
    df_eff_fort_baseline = df_eff_fort_baseline[(df_eff_fort_baseline.sex == "Female")]

if "age_start" in df_eff_fort_baseline.columns:
    df_eff_fort_baseline = df_eff_fort_baseline[(df_eff_fort_baseline.age_start >= 15) & (df_eff_fort_baseline.age_end <= 50)]

In [38]:
if "sex" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[(df_eff_fort_intervention.sex == "Female")]

if "age_start" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[(df_eff_fort_intervention.age_start >= 15) & (df_eff_fort_intervention.age_end <= 50)]

In [39]:
def aggregate_using_population(effective_fort):
    merge_cols = [c for c in ["sex", "wealth_quintile", "age_start", "age_end"] if c in effective_fort.columns]
    merged = effective_fort.merge(population.reset_index(), on=[c for c in merge_cols if "age" not in c], suffixes=("_fort", "_pop"))
    assert ("age_start" in merge_cols) == ("age_end" in merge_cols)
    if "age_start" in merge_cols:
        merged = merged[(merged.age_start_pop >= merged.age_start_fort) & (merged.age_end_pop <= merged.age_end_fort)]
    print(merged)
    assert len(merged) == len(population)
    return merged.groupby(["wealth_quintile"]).apply(lambda df: (df.value_fort * df.value_pop).sum() / df.value_pop.sum())

In [40]:
df_eff_fort_baseline = aggregate_using_population(df_eff_fort_baseline)
df_eff_fort_baseline

   vehicle_name wealth_quintile  value_fort  index     sex  age_start  \
0      bouillon          fourth         0.0     40  Female       15.0   
1      bouillon          fourth         0.0     45  Female       20.0   
2      bouillon          fourth         0.0     50  Female       25.0   
3      bouillon          fourth         0.0     55  Female       30.0   
..          ...             ...         ...    ...     ...        ...   
31     bouillon          second         0.0     59  Female       30.0   
32     bouillon          second         0.0     64  Female       35.0   
33     bouillon          second         0.0     69  Female       40.0   
34     bouillon          second         0.0     74  Female       45.0   

    age_end     value_pop  
0      20.0  3.035520e+06  
1      25.0  2.413412e+06  
2      30.0  1.977992e+06  
3      35.0  1.539460e+06  
..      ...           ...  
31     35.0  1.346347e+06  
32     40.0  1.106389e+06  
33     45.0  9.040506e+05  
34     50.0  7.61

wealth_quintile
fourth     0.0
highest    0.0
lowest     0.0
middle     0.0
second     0.0
dtype: float64

In [41]:
df_eff_fort_intervention = aggregate_using_population(df_eff_fort_intervention)
df_eff_fort_intervention

   vehicle_name wealth_quintile     sex  value_fort  index  age_start  \
0      bouillon          lowest  Female    0.618069     42       15.0   
1      bouillon          lowest  Female    0.618069     47       20.0   
2      bouillon          lowest  Female    0.618069     52       25.0   
3      bouillon          lowest  Female    0.618069     57       30.0   
..          ...             ...     ...         ...    ...        ...   
31     bouillon         highest  Female    0.629615     56       30.0   
32     bouillon         highest  Female    0.629615     61       35.0   
33     bouillon         highest  Female    0.629615     66       40.0   
34     bouillon         highest  Female    0.629615     71       45.0   

    age_end     value_pop  
0      20.0  2.337059e+06  
1      25.0  1.991644e+06  
2      30.0  1.649956e+06  
3      35.0  1.361188e+06  
..      ...           ...  
31     35.0  1.826316e+06  
32     40.0  1.528162e+06  
33     45.0  1.270936e+06  
34     50.0  1.08

wealth_quintile
fourth     0.632860
highest    0.629615
lowest     0.618069
middle     0.627742
second     0.626300
dtype: float64

In [42]:
quintile_name_map = {
    'lowest': 'lowest',
    'second': 'second',
    'middle': 'middle',
    'fourth': 'fourth',
    'highest': 'highest',
}
df_eff_fort_baseline.index = df_eff_fort_baseline.index.map(quintile_name_map)
df_eff_fort_intervention.index = df_eff_fort_intervention.index.map(quintile_name_map)

In [43]:
RBC_baseline = backcalc_rbc(s_ntd_death_or_stillbirth_count / s_births, 'crider')
RBC_baseline

lowest     542.106217
second     542.106217
middle     542.106217
fourth     542.106217
highest    542.106217
dtype: float64

In [44]:
# Fortification folate needs to be converted into dietary folate equivalents (DFEs)
# for use with our effect size.
# https://www.jandonline.org/article/S0002-8223(00)00027-4/pdf
fortification_mcg_to_dfe = 1.7

In [45]:
s_intervention_folate = s_baseline_folate - (df_eff_fort_baseline * s_daily_vehicle * baseline_concentration_mcg_per_gram * fortification_mcg_to_dfe) + (df_eff_fort_intervention * s_daily_vehicle * intervention_concentration_mcg_per_gram * fortification_mcg_to_dfe)
s_intervention_folate

fourth     329.521379
highest    326.166069
lowest     400.824449
middle     348.110039
second     402.424220
dtype: float64

In [46]:
intevention_folate_pct_increase = (s_intervention_folate - s_baseline_folate) / s_baseline_folate
intevention_folate_pct_increase

fourth     0.623258
highest    0.568106
lowest     1.120764
middle     0.767056
second     1.032446
dtype: float64

In [47]:
RBC_with_fort = RBC_baseline * (1 + ((6/10) * intevention_folate_pct_increase))
RBC_with_fort

fourth     744.829448
highest    726.890527
lowest     906.650187
middle     791.601724
second     877.923310
dtype: float64

In [48]:
def calc_ntd_pr(df, method):
    ln_rbc = np.log(df)
    if method == 'daly':
        ln_odds = 1.6563 - 1.2193*ln_rbc
    elif method == 'crider':
        ln_odds = 4.57 - 1.70*ln_rbc
    p = np.exp(ln_odds) # TODO: better transformation
    return p
s_ntd_death_or_stillbirth_rate_with_fort = calc_ntd_pr(RBC_with_fort, 'crider')
10_000 * s_ntd_death_or_stillbirth_rate_with_fort

fourth     12.654039
highest    13.189504
lowest      9.058957
middle     11.409440
second      9.568626
dtype: float64

In [49]:
s_ntd_death_or_stillbirth_count_with_fort = s_ntd_death_or_stillbirth_rate_with_fort * s_births
s_ntd_death_or_stillbirth_count_with_fort

fourth     2280.268334
highest    2087.552175
lowest     1989.461541
middle     2319.487572
second     2148.803475
dtype: float64

In [50]:
ntd_cases_by_scenario = pd.concat([
    s_ntd_death_or_stillbirth_count.rename("value").rename_axis("wealth_quintile").to_frame().assign(entity="ntd", scenario="baseline").set_index(["entity", "scenario"], append=True).value,
    s_ntd_death_or_stillbirth_count_with_fort.rename("value").rename_axis("wealth_quintile").to_frame().assign(entity="ntd", scenario=intervention_scenario).set_index(["entity", "scenario"], append=True).value,
])
ntd_cases_by_scenario

wealth_quintile  entity  scenario    
lowest           ntd     baseline        4758.801348
second           ntd     baseline        4866.170569
middle           ntd     baseline        4405.223395
fourth           ntd     baseline        3904.784094
                                            ...     
highest          ntd     intervention    2087.552175
lowest           ntd     intervention    1989.461541
middle           ntd     intervention    2319.487572
second           ntd     intervention    2148.803475
Name: value, Length: 10, dtype: float64

In [51]:
path = f'./results/{location}/{vehicle}/{intervention_scenario}/ntd_cases_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)

In [52]:
# For calculating YLLs
tmrle = vivarium_inputs.get_theoretical_minimum_risk_life_expectancy()
tmrle

,,value
age_start,age_end,
0.00,0.01,89.958040
0.01,0.02,89.975474
0.02,0.03,89.990990
0.03,0.04,89.985077
...,...,...
109.97,109.98,4.509941
109.98,109.99,4.504631
109.99,110.00,4.499321
110.00,125.00,4.494011


In [53]:
# NOTE: Treating stillbirths as a death!
yll_per_ntd = float(tmrle.iloc[0])
yll_per_ntd

89.95803974533831

In [54]:
ylls_by_scenario = ntd_cases_by_scenario * yll_per_ntd
ylls_by_scenario

wealth_quintile  entity  scenario    
lowest           ntd     baseline        428092.440821
second           ntd     baseline        437751.165498
middle           ntd     baseline        396285.261284
fourth           ntd     baseline        351266.722707
                                             ...      
highest          ntd     intervention    187792.101519
lowest           ntd     intervention    178968.060340
middle           ntd     intervention    208656.555221
second           ntd     intervention    193302.148375
Name: value, Length: 10, dtype: float64

In [55]:
path = f'./results/{location}/{vehicle}/{intervention_scenario}/ylls_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylls_by_scenario.to_csv(path)